# 01 Extraction

This notebook rebuilds the initial NIFTY-50 trading dataset from the committed raw CSV files. It detects stock-market files versus metadata files, combines the selected stock files into one master table, attaches company and industry metadata when available, profiles the extracted dataset, and saves the result to `data/processed/nifty50_combined_raw.csv`.

In [1]:
from pathlib import Path
import sys

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "scripts").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from scripts.etl_pipeline import load_raw_stock_files

RAW_DIR = REPO_ROOT / "data" / "raw"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


## Detect Raw Files

The pipeline checks every CSV inside `data/raw/`, classifies it as a stock-trading file, a combined stock file, or a metadata file, and then prefers per-stock files when both per-stock and already-combined files are present. That keeps the extraction reproducible without double-counting rows.

In [2]:
combined_raw, metadata_df, extraction_report = load_raw_stock_files(RAW_DIR)

print("Extraction strategy:", extraction_report["selection_strategy"])
print("CSV files found:", extraction_report["csv_files_found"])
print("Per-stock files detected:", extraction_report["stock_files_found"])
print("Combined stock files detected:", extraction_report["combined_stock_files_found"])
print("Metadata file:", extraction_report["metadata_file"])
print("Rows with metadata:", extraction_report["rows_with_metadata"])
print("Rows without metadata:", extraction_report["rows_without_metadata"])


Extraction strategy: individual_stock_files
CSV files found: 52
Per-stock files detected: 50
Combined stock files detected: 1
Metadata file: data/raw/nifty-dataset/stock_metadata.csv
Rows with metadata: 235192
Rows without metadata: 0


In [3]:
pd.Series(extraction_report["selected_stock_files"], name="selected_stock_file")

0     data/raw/nifty-dataset/ADANIPORTS.csv
1     data/raw/nifty-dataset/ASIANPAINT.csv
2       data/raw/nifty-dataset/AXISBANK.csv
3     data/raw/nifty-dataset/BAJAJ-AUTO.csv
4     data/raw/nifty-dataset/BAJAJFINSV.csv
5     data/raw/nifty-dataset/BAJFINANCE.csv
6     data/raw/nifty-dataset/BHARTIARTL.csv
7           data/raw/nifty-dataset/BPCL.csv
8      data/raw/nifty-dataset/BRITANNIA.csv
9          data/raw/nifty-dataset/CIPLA.csv
10     data/raw/nifty-dataset/COALINDIA.csv
11       data/raw/nifty-dataset/DRREDDY.csv
12     data/raw/nifty-dataset/EICHERMOT.csv
13          data/raw/nifty-dataset/GAIL.csv
14        data/raw/nifty-dataset/GRASIM.csv
15       data/raw/nifty-dataset/HCLTECH.csv
16          data/raw/nifty-dataset/HDFC.csv
17      data/raw/nifty-dataset/HDFCBANK.csv
18    data/raw/nifty-dataset/HEROMOTOCO.csv
19      data/raw/nifty-dataset/HINDALCO.csv
20    data/raw/nifty-dataset/HINDUNILVR.csv
21     data/raw/nifty-dataset/ICICIBANK.csv
22    data/raw/nifty-dataset/IND

## Dataset Profile

This summary gives us the starting shape, column structure, date coverage, and missing-value counts before the cleaning notebook standardizes field names and applies data-quality rules.

In [4]:
date_series = pd.to_datetime(combined_raw["Date"], errors="coerce")
missing_summary = combined_raw.isna().sum().sort_values(ascending=False)

print("Shape:", combined_raw.shape)
print("Columns:", list(combined_raw.columns))
print("Date range:", date_series.min(), "to", date_series.max())
print("Distinct row-level symbols:", combined_raw["Symbol"].nunique())
print("Distinct source symbols:", combined_raw["source_symbol"].nunique())
missing_summary[missing_summary > 0]

Shape: (235192, 21)
Columns: ['Date', 'Symbol', 'Series', 'Prev Close', 'Open', 'High', 'Low', 'Last', 'Close', 'VWAP', 'Volume', 'Turnover', 'Trades', 'Deliverable Volume', '%Deliverble', 'source_file', 'source_symbol', 'Company Name', 'Industry', 'ISIN Code', 'metadata_join_key']
Date range: 2000-01-03 00:00:00 to 2021-04-30 00:00:00
Distinct row-level symbols: 65
Distinct source symbols: 49


Trades                114848
%Deliverble            16077
Deliverable Volume     16077
dtype: int64

In [5]:
combined_raw.head()

,Date,Symbol,Series,Prev Close,Open,High,Low,Last,Close,VWAP,...,Turnover,Trades,Deliverable Volume,%Deliverble,source_file,source_symbol,Company Name,Industry,ISIN Code,metadata_join_key
0,2007-11-27,MUNDRAPORT,EQ,440.0,770.0,1050.0,770.0,959.0,962.9,984.72,...,2687719053785000.0,NaN,9859619,0.3612,ADANIPORTS.csv,ADANIPORTS,Adani Ports and Special Economic Zone Ltd.,SERVICES,INE742F01042,source_symbol
1,2007-11-28,MUNDRAPORT,EQ,962.9,984.0,990.0,874.0,885.0,893.9,941.38,...,431276530164999.9375,NaN,1453278,0.3172,ADANIPORTS.csv,ADANIPORTS,Adani Ports and Special Economic Zone Ltd.,SERVICES,INE742F01042,source_symbol
2,2007-11-29,MUNDRAPORT,EQ,893.9,909.0,914.75,841.0,887.0,884.2,888.09,...,455065846264999.9375,NaN,1069678,0.2088,ADANIPORTS.csv,ADANIPORTS,Adani Ports and Special Economic Zone Ltd.,SERVICES,INE742F01042,source_symbol
3,2007-11-30,MUNDRAPORT,EQ,884.2,890.0,958.0,890.0,929.0,921.55,929.17,...,428325662830000.0,NaN,1260913,0.2735,ADANIPORTS.csv,ADANIPORTS,Adani Ports and Special Economic Zone Ltd.,SERVICES,INE742F01042,source_symbol
4,2007-12-03,MUNDRAPORT,EQ,921.55,939.75,995.0,922.0,980.0,969.3,965.65,...,287519974300000.0,NaN,816123,0.2741,ADANIPORTS.csv,ADANIPORTS,Adani Ports and Special Economic Zone Ltd.,SERVICES,INE742F01042,source_symbol


In [6]:
output_path = PROCESSED_DIR / "nifty50_combined_raw.csv"
combined_raw.to_csv(output_path, index=False)
print(f"Saved extracted dataset to {output_path}")

Saved extracted dataset to /Users/devaanshkathuria/Documents/Projects/DVA_Final/DVA_capstone_2/data/processed/nifty50_combined_raw.csv
